# Incendios en España — pipeline de satélites y viento

Este cuaderno solo importa y llama a las funciones del paquete `incendios/` y muestra el resultado. Todo el crawling se hace con **crawl4ai**, y solo se guarda lo que cae en **España**: se filtra con el contorno real del país, no con una caja.

**Uso:**
1. Pega tus claves en la sección 1.
2. Ejecuta todo el cuaderno con *Run All*.

Cada fuente funciona aislada. Si falla o le falta la clave, se anota en el resumen final y el resto sigue.

| Sección | Fuente | ¿Necesita clave? |
|---|---|---|
| 4.1 | NASA FIRMS — VIIRS y MODIS (focos de calor) | no (opcional) |
| 4.2 | EUMETSAT LSA-SAF — SEVIRI (focos cada 15 min) | sí: usuario + contraseña |
| 4.3 | EFFIS — áreas quemadas | no |
| 4.4 | Incidencias 112 oficiales (INFOCA, FIDIAS, Bombers, INFORCYL) | no |
| 4.5 | Riesgo de incendio previsto por AEMET (mapa oficial georreferenciado) | **sí: API key de AEMET** |
| 4.6 | ESA WorldCover — cobertura del suelo en cada foco (local) | no (generar una vez) |
| 5.1 | AEMET — ~855 estaciones (viento, temperatura, humedad) | **sí: API key** |
| 5.2 | Puertos del Estado (iMar) — ~100 boyas y estaciones de puerto | no |
| 5.3 | Open-Meteo — viento en las 52 capitales | no |
| 5.4 | Open-Meteo — viento en cada foco | no |

## 0. Preparación

In [1]:
# ============================================================================
# Ruta del proyecto: localizar la carpeta incendios/ desde el notebook
# ============================================================================
import sys
from pathlib import Path

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "incendios" / "__init__.py").exists()), None)
if RAIZ is None:
    raise FileNotFoundError("No encuentro la carpeta incendios/: abre el notebook desde Incendios-V0.0/")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
print("Proyecto:", RAIZ)

# Si falta algo, descomenta la siguiente línea y ejecútala una vez:
# %pip install -r {RAIZ}/requirements.txt

Proyecto: /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0


In [2]:
# Recarga automática: si el código de incendios/ cambia, se usa la versión nueva
# sin reiniciar el kernel.
%load_ext autoreload
%autoreload 2

import pandas as pd

from incendios import config
from incendios.config import fijar_claves, estado_claves
from incendios.verificacion import verificar_claves
from incendios.pipeline import a_geojson, anotar, ejecutar, guardar, nueva_salida, resumen
from incendios import cobertura
from incendios.satelites import effis, firms, lsasaf
from incendios.viento import aemet, openmeteo, puertos
from incendios import visor, riesgo
from incendios import incidencias as incid
from incendios.pipeline import ejecutar_con_detalle

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Claves

Pega cada clave **entre las comillas**. Las que dejes vacías se ignoran, y en ese caso se usa lo que haya en `.env` si existe. Si ya guardaste tus claves en `.env`, deja todo vacío.

| Clave | Para qué | Alta gratuita |
|---|---|---|
| `AEMET_API_KEY` | estaciones de AEMET (viento observado) | https://opendata.aemet.es/centrodedescargas/altaUsuario |
| `FIRMS_MAP_KEY` | opcional: sin ella FIRMS usa los CSV públicos | https://firms.modaps.eosdis.nasa.gov/api/map_key/ |
| `LSASAF_USUARIO` / `LSASAF_PASSWORD` | focos SEVIRI cada 15 min | https://mokey.lsasvcs.ipma.pt/auth/signup |

⚠️ **Si pegas las claves aquí, no subas este notebook a git con ellas.** Lo más cómodo es lo siguiente:
1. Pon `GUARDAR_EN_ENV = True` y ejecuta la celda una vez. Las claves se guardan en `.env`, que git ignora.
2. Vacía las comillas.

A partir de ahí se leen solas de `.env`.

In [3]:
# ============================================================================
# PEGA AQUÍ TUS CLAVES (entre las comillas)
# ============================================================================
AEMET_API_KEY   = ""
FIRMS_MAP_KEY   = ""   # opcional
LSASAF_USUARIO  = ""   # opcional (SEVIRI)
LSASAF_PASSWORD = ""   # opcional (SEVIRI)

GUARDAR_EN_ENV = False   # True → las escribe en .env para no volver a pegarlas
# ============================================================================

fijar_claves(aemet=AEMET_API_KEY, firms=FIRMS_MAP_KEY,
             lsasaf_usuario=LSASAF_USUARIO, lsasaf_password=LSASAF_PASSWORD,
             guardar_en_env=GUARDAR_EN_ENV)

# Nunca se imprime una clave entera: solo inicio, final y longitud.
for variable, valor in estado_claves().items():
    print(f"{variable:<16} {valor}")
if GUARDAR_EN_ENV:
    print("\nGuardadas en", RAIZ / ".env", "→ ya puedes vaciar las comillas de esta celda.")

AEMET_API_KEY    eyJh…EFOA (284 caracteres)
FIRMS_MAP_KEY    — (vacía)
LSASAF_USER      ivan…te19 (12 caracteres)
LSASAF_PASSWORD  ****


## 2. Verificar claves

Hace una petición mínima a cada servicio para saber, **antes de lanzar el pipeline**, si la clave pegada funciona.

In [4]:
for servicio, resultado in (await verificar_claves()).items():
    print(f"{servicio:<18} {resultado}")

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

AEMET              OK
NASA FIRMS         omitida: se usan los CSV públicos (funciona igual, sin clave)
EUMETSAT LSA-SAF   OK


## 3. Parámetros

Ventanas de tiempo. `estado` acumula qué fuente fue bien y cuál no; `SALIDA` es la carpeta de esta ejecución.

In [5]:
HORAS_FOCOS     = 24     # FIRMS: 24, 48 o hasta 168 (7 días) sin clave; hasta 120 con MAP_KEY
DIAS_INCIDENCIAS = 3     # incidencias oficiales ya extinguidas que se muestran
HORAS_SEVIRI    = 3      # SEVIRI: 4 ficheros por hora, mejor ventanas cortas
DIAS_EFFIS      = 7      # áreas quemadas
VIENTO_EN_FOCOS = True   # consultar el viento en el punto de cada foco
ABRIR_VISOR     = True   # abrir visor.html en el navegador al terminar (sección 8)

estado = {}
SALIDA = nueva_salida()
print("Salida de esta ejecución:", SALIDA)

Salida de esta ejecución: /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0/output/20261007T095415Z


## 4. Satélites

### 4.1 NASA FIRMS — VIIRS (375 m) y MODIS (1 km)

Los satélites van en órbita polar, así que cada uno pasa sobre España un par de veces al día. `confianza` está unificada en alta/media/baja y `frp_mw` es la potencia radiativa del fuego.

In [6]:
focos_firms = await ejecutar(estado, "NASA FIRMS", firms.obtener_focos(HORAS_FOCOS))
print(len(focos_firms), "focos en España en las últimas", HORAS_FOCOS, "h")

df_firms = pd.DataFrame(focos_firms)
if not df_firms.empty:
    display(df_firms.groupby(["satelite", "confianza"]).size().unstack(fill_value=0))
    display(df_firms.sort_values("frp_mw", ascending=False).head(10))

[INIT].... → Crawl4AI 0.8.9 

18 focos en España en las últimas 24 h


confianza,baja,media
satelite,,
Aqua,1,0
NOAA-20,0,7
NOAA-21,0,4
Suomi NPP,0,6


,fuente,sensor,satelite,lat,lon,fecha_utc,confianza,confianza_bruta,frp_mw,dia_noche,resolucion_m
17,NASA FIRMS,MODIS,Aqua,38.58708,-5.22434,2026-10-06T15:52:00+00:00,baja,48,14.33,dia,1000
14,NASA FIRMS,VIIRS,NOAA-21,39.63929,-6.97539,2026-10-06T13:10:00+00:00,media,nominal,5.53,dia,375
15,NASA FIRMS,VIIRS,NOAA-21,39.63838,-6.98035,2026-10-06T13:10:00+00:00,media,nominal,5.53,dia,375
0,NASA FIRMS,VIIRS,Suomi NPP,40.24523,-3.47569,2026-10-07T02:06:00+00:00,media,nominal,3.63,noche,375
13,NASA FIRMS,VIIRS,NOAA-21,39.81620,-5.66091,2026-10-06T13:10:00+00:00,media,nominal,3.50,dia,375
16,NASA FIRMS,VIIRS,NOAA-21,40.24366,-3.47550,2026-10-06T13:10:00+00:00,media,nominal,3.48,dia,375
6,NASA FIRMS,VIIRS,NOAA-20,40.24595,-3.47153,2026-10-06T14:08:00+00:00,media,nominal,2.39,dia,375
10,NASA FIRMS,VIIRS,NOAA-20,36.96652,-1.90376,2026-10-07T02:26:00+00:00,media,nominal,1.56,noche,375
3,NASA FIRMS,VIIRS,Suomi NPP,36.96669,-1.90653,2026-10-07T02:06:00+00:00,media,nominal,1.29,noche,375
1,NASA FIRMS,VIIRS,Suomi NPP,39.93721,-3.74500,2026-10-07T02:06:00+00:00,media,nominal,1.11,noche,375


### 4.2 EUMETSAT LSA-SAF — SEVIRI (Meteosat, cada 15 min)

Este satélite es geoestacionario: ve España las 24 h y cubre los huecos entre pasadas de FIRMS, aunque con peor resolución (~3–5 km). Necesita usuario y contraseña; sin ellos aparece como `omitida`.

⚠️ **SEVIRI no detecta fuego bajo las nubes.** Por eso, además de los focos, se calcula **qué porcentaje de España pudo vigilar** en el fichero más reciente:
- **Con `pct_vigilado` bajo,** "0 focos" significa *"no se pudo mirar"*, no *"no hay fuego"*.
- **En días nublados,** la referencia es FIRMS: MODIS y VIIRS ven mejor entre nubes finas.

In [7]:
focos_seviri = await ejecutar(estado, "EUMETSAT LSA-SAF (SEVIRI)", lsasaf.obtener_focos(HORAS_SEVIRI))
await anotar(estado, "EUMETSAT LSA-SAF (SEVIRI)", "vigilancia", lsasaf.vigilancia_espana())

e = estado["EUMETSAT LSA-SAF (SEVIRI)"]
print(e.get("estado"), "·", e.get("registros", e.get("detalle")), "focos en las últimas", HORAS_SEVIRI, "h")
v = e.get("vigilancia")
if isinstance(v, dict):
    print(f"Vigilancia a las {v['fecha_utc'][11:16]} UTC: {v['pct_vigilado']}% de España vigilado · "
          f"{v['pct_nubes']}% bajo nubes · {v['pct_otros']}% no procesable (agua, costa, urbano)")
    if v["pct_vigilado"] < 30:
        print("→ Cobertura baja: un resultado de 0 focos NO descarta incendios.")
if focos_seviri:
    display(pd.DataFrame(focos_seviri).sort_values("fecha_utc", ascending=False).head(10))

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

ok · 0 focos en las últimas 3 h
Vigilancia a las 09:15 UTC: 16.7% de España vigilado · 63.3% bajo nubes · 20.0% no procesable (agua, costa, urbano)
→ Cobertura baja: un resultado de 0 focos NO descarta incendios.


### 4.3 EFFIS — áreas quemadas (Copernicus)

Son perímetros de lo ya quemado, cartografiados con MODIS y Sentinel-2. Llegan con días de retraso, pero son la cifra de referencia europea.

La ventana cuenta lo que EFFIS **publicó o actualizó** en esos días, no la fecha del incendio, porque EFFIS sigue corrigiendo perímetros después. La columna `novedad` distingue entre:
- **`nueva`:** el incendio cae dentro de la ventana.
- **`actualizada`:** incendio anterior cuyo perímetro EFFIS ha revisado.

In [8]:
areas = await ejecutar(estado, "EFFIS (áreas quemadas)", effis.obtener_areas(DIAS_EFFIS))
print(len(areas), "áreas quemadas publicadas o actualizadas en los últimos", DIAS_EFFIS, "días")
if areas:
    df_areas = pd.DataFrame(areas)
    df_areas["fecha_incendio"] = df_areas["fecha_incendio"].str[:10]
    df_areas["ultima_actualizacion"] = df_areas["ultima_actualizacion"].str[:10]
    print(df_areas["novedad"].value_counts().to_dict(), "·", df_areas["area_ha"].sum(), "ha en total")
    display(df_areas[["fecha_incendio", "novedad", "ultima_actualizacion", "provincia", "municipio",
                      "area_ha", "pct_natura2000"]].sort_values("area_ha", ascending=False))

[INIT].... → Crawl4AI 0.8.9 

26 áreas quemadas publicadas o actualizadas en los últimos 7 días
{'actualizada': 26} · 196.0 ha en total


,fecha_incendio,novedad,ultima_actualizacion,provincia,municipio,area_ha,pct_natura2000
22,2026-09-28,actualizada,2026-10-01,Toledo,Lominchar,33.0,0.0
2,2026-09-21,actualizada,2026-10-05,Zamora,Muelas del Pan,26.0,0.0
20,2026-09-25,actualizada,2026-10-01,Segovia,Ventosilla y Tejadilla,21.0,0.0
24,2026-09-26,actualizada,2026-10-01,Asturias,Lena,18.0,100.0
21,2026-09-28,actualizada,2026-10-01,Toledo,Chozas de Canales,14.0,0.0
1,2026-09-28,actualizada,2026-10-05,Cantabria,Campoo de Enmedio,10.0,0.0
23,2026-09-25,actualizada,2026-10-01,Ourense,Manzaneda,9.0,100.0
8,2026-09-24,actualizada,2026-10-05,Cantabria,"Tojos, Los",9.0,100.0
9,2026-09-24,actualizada,2026-10-05,Cantabria,"Tojos, Los",9.0,100.0
25,2026-09-26,actualizada,2026-10-01,Asturias,Lena,7.0,100.0


### 4.4 Incidencias 112 oficiales

Son los incendios que declaran los servicios de emergencia autonómicos:
- **Andalucía:** INFOCA.
- **Castilla-La Mancha:** INFOCAM/FIDIAS.
- **Cataluña:** Bombers.
- **Castilla y León:** INFORCYL.

Se muestran los que no están extinguidos y los extinguidos de los últimos `DIAS_INCIDENCIAS` días. La Comunitat Valenciana no publica un servicio de datos abierto.

In [9]:
oficiales = await ejecutar_con_detalle(estado, "Incidencias 112 (oficiales)", incid.obtener_incidencias(DIAS_INCIDENCIAS))
print(estado["Incidencias 112 (oficiales)"].get("detalle_fuentes"))
if oficiales:
    display(pd.DataFrame(oficiales)[["comunidad", "municipio", "provincia", "estado", "inicio_utc", "superficie_ha", "fuente"]])

[INIT].... → Crawl4AI 0.8.9 

{'Andalucía (INFOCA)': 13, 'Castilla-La Mancha (FIDIAS)': 0, 'Cataluña (Bombers)': 1, 'Castilla y León (INFORCYL)': 0}


,comunidad,municipio,provincia,estado,inicio_utc,superficie_ha,fuente
0,Cataluña,Montblanc,NaN,aviso sin fase,2026-10-07T09:33+00:00,None,Bombers de la Generalitat
1,Andalucía,Casares,MÁLAGA,extinguido,2026-10-06T19:40+00:00,None,INFOCA
2,Andalucía,Paterna del Campo,HUELVA,extinguido,2026-10-06T17:05+00:00,None,INFOCA
3,Andalucía,Calañas,HUELVA,extinguido,2026-10-06T13:40+00:00,None,INFOCA
4,Andalucía,El Almendro,HUELVA,extinguido,2026-10-06T13:00+00:00,None,INFOCA
5,Andalucía,Alhaurín el Grande,MÁLAGA,extinguido,2026-10-05T20:44+00:00,None,INFOCA
6,Andalucía,Guillena,SEVILLA,extinguido,2026-10-05T05:54+00:00,None,INFOCA
7,Andalucía,Cortes y Graena,GRANADA,extinguido,2026-10-04T16:50+00:00,None,INFOCA
8,Andalucía,Níjar,ALMERÍA,extinguido,2026-10-04T14:40+00:00,None,INFOCA
9,Andalucía,Villalba del Alcor,HUELVA,extinguido,2026-10-04T14:08+00:00,None,INFOCA


### 4.5 Riesgo de incendio previsto (AEMET)

AEMET publica cada día un mapa oficial del riesgo meteorológico de incendio para mañana, pasado y el día siguiente. Lo publica solo como imagen, así que se georreferencia (96 % de coincidencia con el contorno de España) y se reclasifica en sus 6 niveles, de "muy bajo" a "extremo".

In [10]:
prevision = await ejecutar(estado, "Riesgo de incendio (AEMET)", riesgo.obtener_prevision())
for p in prevision:
    print(f"{p['fecha']}: nivel máximo {p['nivel_max']} · reparto (%) {p['reparto_pct']}")

[INIT].... → Crawl4AI 0.8.9 

2026-10-08: nivel máximo Alto · reparto (%) {'Muy bajo': 85.2, 'Bajo': 14.1, 'Moderado': 0.6, 'Alto': 0.0, 'Muy alto': 0.0, 'Extremo': 0.0}
2026-10-09: nivel máximo Alto · reparto (%) {'Muy bajo': 75.1, 'Bajo': 23.8, 'Moderado': 1.1, 'Alto': 0.0, 'Muy alto': 0.0, 'Extremo': 0.0}
2026-10-10: nivel máximo Moderado · reparto (%) {'Muy bajo': 69.7, 'Bajo': 30.2, 'Moderado': 0.1, 'Alto': 0.0, 'Muy alto': 0.0, 'Extremo': 0.0}


### 4.6 Cobertura del suelo en cada foco (ESA WorldCover)

Qué hay en el suelo alrededor de cada foco y de cada incidencia: bosque, matorral, pasto, cultivo, urbano, agua… Se mira una ventana del tamaño del píxel del satélite (~375 m en VIIRS, ~1 km en MODIS).
- **En el visor:** las Acciones distinguen probables quemas agrícolas, fuentes urbanas o industriales y falsos positivos sobre agua de la vegetación forestal. La propagación solo se dibuja donde hay vegetación natural.
- **Sin red:** usa datos locales. Si aparece `omitida`, hay que generarlos una vez con `python herramientas/generar_cobertura.py`, que descarga 1,4 GB.

In [ ]:
await ejecutar(estado, cobertura.NOMBRE_FUENTE, cobertura.anotar_async(focos_firms + focos_seviri, oficiales))
print(estado[cobertura.NOMBRE_FUENTE])
filas = [{"fecha_utc": f["fecha_utc"], "satelite": f["satelite"], "frp_mw": f["frp_mw"],
          "suelo dominante": f["cobertura"]["dominante"], "vegetación natural %": f["cobertura"]["combustible_pct"],
          "reparto %": f["cobertura"]["fracciones_pct"]}
         for f in focos_firms + focos_seviri if f.get("cobertura")]
if filas:
    display(pd.DataFrame(filas).sort_values("frp_mw", ascending=False))

## 5. Viento

Cada lectura incluye:
- `direccion_grados`: de dónde viene el viento.
- `hacia_grados`: hacia dónde empuja el fuego.
- `regla_30`: indica si se cumplen >30 °C, <30 % de humedad y >30 km/h.
- `antiguedad_min`: minutos desde la medida. No se descarta ninguna estación por ir atrasada; esta columna dice cuánto fiarse.

### 5.1 AEMET — estaciones automáticas (capa nacional, observación real)

Se guarda la última lectura de cada estación. El viento llega en m/s y se convierte a km/h. Se incluyen también las ~100 estaciones sin anemómetro, porque su temperatura y su humedad cuentan para el riesgo de incendio.

In [11]:
estaciones = await ejecutar(estado, "AEMET (estaciones)", aemet.obtener_estaciones())
print(estado["AEMET (estaciones)"])

columnas_viento = ["nombre", "antiguedad_min", "velocidad_kmh", "racha_kmh", "direccion_cardinal",
                   "temperatura_c", "humedad_pct", "regla_30"]
if estaciones:
    df_aemet = pd.DataFrame(estaciones)
    print("Sin anemómetro (solo temperatura/humedad):", int(df_aemet["velocidad_kmh"].isna().sum()),
          "· lectura de hace más de 3 h:", int((df_aemet["antiguedad_min"] > 180).sum()),
          "· cumplen la regla del 30:", int(df_aemet["regla_30"].fillna(False).sum()))
    display(df_aemet.sort_values("racha_kmh", ascending=False)[columnas_viento].head(15))

[INIT].... → Crawl4AI 0.8.9 

{'estado': 'ok', 'registros': 853, 'segundos': 1.7}
Sin anemómetro (solo temperatura/humedad): 107 · lectura de hace más de 3 h: 60 · cumplen la regla del 30: 0


,nombre,antiguedad_min,velocidad_kmh,racha_kmh,direccion_cardinal,temperatura_c,humedad_pct,regla_30
101,CABO PEÑAS,55,43.9,64.1,O,16.1,93.0,False
84,CORISCAO · PARQUE NACIONAL PICOS DE EUROPA,55,34.6,54.7,O,8.2,89.0,False
112,CABO BUSTO,55,36.4,51.1,ONO,17.0,87.0,False
93,EVC_NIEMBRU-LLANES,55,30.2,50.8,OSO,16.7,84.0,False
102,ASTURIAS/AVILÉS,55,32.4,50.0,O,15.2,95.0,False
73,SAN ROQUE DE RIOMERA-CARACOL,115,19.4,49.3,ONO,12.2,100.0,False
748,CABRERA · PARQUE NACIONAL DE CABRERA,55,27.7,46.8,S,23.8,80.0,False
75,SANTANDER CMT,55,28.8,46.8,O,19.5,77.0,False
796,ARURE CEMENTERIO,175,27.4,44.3,ENE,15.5,97.0,False
88,VEGA DE URRIELLU · PARQUE NACIONAL PICOS DE EU...,55,15.8,43.9,ENE,6.7,87.0,False


### 5.2 Puertos del Estado — boyas, mareógrafos y estaciones de puerto (iMar)

Es la red de la app iMar (PORTUS). Cubre la costa y mar adentro, donde AEMET apenas tiene estaciones. No necesita clave.

Solo se guardan las estaciones españolas: el visor también muestra boyas de Irlanda, Reino Unido, Francia y Portugal, y esas se descartan.

In [12]:
costa = await ejecutar(estado, "Puertos del Estado (costa)", puertos.obtener_estaciones())
if costa:
    df_costa = pd.DataFrame(costa)
    display(df_costa.groupby(["red", "tipo_estacion"]).size().rename("estaciones").to_frame())
    display(df_costa.sort_values("racha_kmh", ascending=False)[["nombre", "tipo_estacion"] + columnas_viento[1:6]].head(10))

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

,,estaciones
red,tipo_estacion,
Estaciones meteorologicas,estación de puerto,60
Mareografos,mareógrafo,26
Red costera,boya,3
Red exterior,boya,15


,nombre,tipo_estacion,antiguedad_min,velocidad_kmh,racha_kmh,direccion_cardinal,temperatura_c
63,Estación Meteorológica Morro de dique de Abrigo,estación de puerto,15,27.7,40.3,O,NaN
69,Estación Meteorológica Endesa,estación de puerto,15,27.4,38.9,O,NaN
65,Est. Meteorológica de Dique de Abrigo,estación de puerto,15,27.7,38.9,O,23.2
81,Estac. Meteo. Sagunto Norte,estación de puerto,5,12.6,38.5,SO,25.3
80,Estac. Meteo. Sagunto Muelle de Levante,estación de puerto,5,26.3,38.2,OSO,22.2
35,Mareógrafo de Almería,mareógrafo,1,34.9,37.8,OSO,NaN
38,Mareógrafo de Tarragona,mareógrafo,8,30.6,34.2,ONO,NaN
75,Estación Meteorológica Faro de Almería,estación de puerto,3,31.0,33.8,OSO,24.1
66,Est. Meteorológica de Pta. Carnero,estación de puerto,15,21.6,32.4,SSO,NaN
34,Mareógrafo de Motril 2,mareógrafo,1,24.1,32.0,OSO,NaN


### 5.3 Open-Meteo — capitales de provincia (respaldo, sin clave)

Es salida de modelo, no medida. Sirve para tener una visión nacional aunque no haya clave de AEMET.

In [13]:
rejilla = await ejecutar(estado, "Open-Meteo (capitales)", openmeteo.obtener_rejilla())
if rejilla:
    display(pd.DataFrame(rejilla).sort_values("racha_kmh", ascending=False)[columnas_viento].head(15))

[INIT].... → Crawl4AI 0.8.9 

,nombre,antiguedad_min,velocidad_kmh,racha_kmh,direccion_cardinal,temperatura_c,humedad_pct,regla_30
5,Santander,10,31.2,60.8,OSO,19.8,71.0,False
22,Palma,10,15.3,37.8,SO,23.7,73.0,False
19,Valencia,10,14.1,32.8,OSO,24.2,57.0,False
50,Ceuta,10,17.3,32.0,O,22.7,68.0,False
11,Zaragoza,10,14.1,31.7,ONO,18.5,81.0,False
48,Santa Cruz de Tenerife,10,15.5,31.3,N,24.8,62.0,False
49,Las Palmas de Gran Canaria,10,15.1,30.2,N,24.1,70.0,False
37,Albacete,10,11.8,29.9,ONO,17.4,78.0,False
26,Soria,10,15.5,29.5,O,14.5,78.0,False
35,Toledo,10,13.0,28.8,O,17.0,85.0,False


### 5.4 Viento en cada foco (capa de emergencia)

Viento en la celda de ~10 km de cada foco. Es lo más parecido al visor ALCIF de AEMET, que no tiene API pública.

In [14]:
focos = sorted(focos_firms + focos_seviri, key=lambda f: f["fecha_utc"], reverse=True)

if focos and VIENTO_EN_FOCOS:
    por_celda = dict(await ejecutar(estado, "Open-Meteo (viento en focos)", openmeteo.viento_en_focos(focos)))
    for f in focos:
        f["viento_local"] = por_celda.get(openmeteo.celda_de(f))

    df_focos = pd.DataFrame([{**{k: f[k] for k in ("fecha_utc", "satelite", "lat", "lon", "confianza", "frp_mw")},
                              **{f"viento_{k}": (f["viento_local"] or {}).get(k)
                                 for k in ("velocidad_kmh", "racha_kmh", "direccion_cardinal", "regla_30")}}
                             for f in focos])
    display(df_focos.sort_values("viento_racha_kmh", ascending=False).head(15))
else:
    print("Sin focos (o VIENTO_EN_FOCOS = False): nada que consultar.")

[INIT].... → Crawl4AI 0.8.9 

,fecha_utc,satelite,lat,lon,confianza,frp_mw,viento_velocidad_kmh,viento_racha_kmh,viento_direccion_cardinal,viento_regla_30
12,2026-10-06T15:52:00+00:00,Aqua,38.58708,-5.22434,baja,14.33,15.3,29.2,O,False
5,2026-10-07T02:26:00+00:00,NOAA-20,36.17740,-5.42310,media,0.68,15.1,28.4,O,False
6,2026-10-07T02:09:00+00:00,Suomi NPP,36.17634,-5.42287,media,0.88,15.1,28.4,O,False
2,2026-10-07T02:26:00+00:00,NOAA-20,39.93861,-3.74664,media,0.99,13.6,24.5,OSO,False
8,2026-10-07T02:06:00+00:00,Suomi NPP,39.93721,-3.74500,media,1.11,13.6,24.5,OSO,False
11,2026-10-07T02:06:00+00:00,Suomi NPP,37.35950,-6.67368,media,0.70,12.9,24.5,NO,False
3,2026-10-07T02:26:00+00:00,NOAA-20,36.96652,-1.90376,media,1.56,13.8,24.1,OSO,False
10,2026-10-07T02:06:00+00:00,Suomi NPP,36.96669,-1.90653,media,1.29,13.8,24.1,OSO,False
14,2026-10-06T13:10:00+00:00,NOAA-21,39.81620,-5.66091,media,3.50,13.4,23.0,O,False
17,2026-10-06T13:10:00+00:00,NOAA-21,40.24366,-3.47550,media,3.48,11.9,22.7,OSO,False


## 6. Guardar y generar el visor

Se guardan los GeoJSON (se abren en QGIS o geojson.io) y se genera **`visor.html`**, el mapa interactivo con todo lo que ha sacado este cuaderno:
- **Menú "Capas":**
  - Relieve 3D, Sombreado, Tintado, Pendiente, Cobertura del suelo, Límites, Riesgo, Incidencias 112, Propagación, Acciones y Exageración.
  - Vientos y Satélites, cada uno con su submenú.
- **Selector de comunidad autónoma:** acerca el mapa y recalcula el parte.
- **Clic en cualquier elemento:** muestra su ficha.

También se escribe `output/visor_publicar.html`, la misma página preparada para publicarla como artifact.

In [15]:
ficheros = [
    guardar(SALIDA, "focos.geojson", a_geojson(focos)),
    guardar(SALIDA, "areas_quemadas.geojson", a_geojson(areas, geometria=lambda a: a.get("geometria"))),
    guardar(SALIDA, "incidencias_oficiales.geojson", a_geojson(oficiales)),
    guardar(SALIDA, "viento_estaciones.geojson", a_geojson(estaciones)),
    guardar(SALIDA, "viento_puertos.geojson", a_geojson(costa)),
    guardar(SALIDA, "viento_rejilla.geojson", a_geojson(rejilla)),
    guardar(SALIDA, "estado_fuentes.json", {"generado_utc": SALIDA.name, "fuentes": estado}),
]
pagina = visor.generar(SALIDA, focos=focos, areas=areas, estaciones=estaciones, costa=costa, rejilla=rejilla,
                       estado=estado, horas_focos=HORAS_FOCOS, dias_effis=DIAS_EFFIS, horas_seviri=HORAS_SEVIRI,
                       incidencias=oficiales, riesgo=prevision, dias_incidencias=DIAS_INCIDENCIAS)

print("Guardado en", SALIDA)
for f in ficheros + [pagina]:
    print("  ", f.name, f"({f.stat().st_size // 1024} KB)")
print()
print("Estado de las fuentes:")
print(resumen(estado))

Guardado en /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0/output/20261007T095415Z
   focos.geojson (15 KB)
   areas_quemadas.geojson (239 KB)
   incidencias_oficiales.geojson (7 KB)
   viento_estaciones.geojson (468 KB)
   viento_puertos.geojson (67 KB)
   viento_rejilla.geojson (28 KB)
   estado_fuentes.json (1 KB)
   visor.html (2543 KB)

Estado de las fuentes:
  ok       NASA FIRMS                           18 registros    2.0s
  ok       EUMETSAT LSA-SAF (SEVIRI)             0 registros    7.1s
                                            vigiló el 16.7% de España (63.3% bajo nubes, 09:15 UTC)
  ok       EFFIS (áreas quemadas)               26 registros   12.0s
  ok       Incidencias 112 (oficiales)          14 registros    1.3s
  ok       Riesgo de incendio (AEMET)            3 registros   11.4s
  ok       AEMET (estaciones)                  853 registros    1.7s
  ok       Puertos del Estado (costa)          104 registros    2.0s
  ok       Open-Meteo (capital

## 7. Comprobar que el visor tiene los datos de esta ejecución

Se lee lo que quedó incrustado dentro de `visor.html` y se compara, fuente por fuente, con lo que acaba de sacar el cuaderno: número de registros, el primer foco y la primera incidencia, la vigilancia de SEVIRI y la hora de generación.
- **Todo ✓:** el visor está al día.
- **Algún ✗:** vuelve a ejecutar la sección 6.

In [16]:
comprobacion = pd.DataFrame(visor.comprobar(pagina, focos=focos, areas=areas, estaciones=estaciones, costa=costa,
                                            rejilla=rejilla, estado=estado, incidencias=oficiales, riesgo=prevision))
comprobacion["ok"] = comprobacion["ok"].map({True: "✓", False: "✗"})
display(comprobacion)

publicar = visor.leer_datos(RAIZ / "output" / "visor_publicar.html")["generado_utc"] == visor.leer_datos(pagina)["generado_utc"]
print("visor_publicar.html es la misma versión:", "✓" if publicar else "✗")
print("Visor al día ✓" if (comprobacion["ok"] == "✓").all() and publicar else "Hay diferencias ✗: vuelve a ejecutar la sección 6")

,comprobación,pipeline,visor,ok
0,Focos de satélite (FIRMS + SEVIRI),18,18,✓
1,Focos con viento en el punto,18,18,✓
2,Áreas quemadas EFFIS,26,26,✓
3,Incidencias 112,14,14,✓
4,Días de riesgo AEMET,3,3,✓
5,Estaciones AEMET,853,853,✓
6,Estaciones Puertos del Estado,104,104,✓
7,Puntos Open-Meteo,52,52,✓
8,Fuentes en el estado,9,9,✓
9,Primer foco idéntico,2026-10-07T02:26:00+00:00,2026-10-07T02:26:00+00:00,✓


visor_publicar.html es la misma versión: ✓
Visor al día ✓


## 8. Abrir el visor

Abre `visor.html` en tu navegador predeterminado. Si `ABRIR_VISOR = False` en la sección 3, solo muestra la ruta para abrirlo a mano.

In [17]:
print("Visor:", pagina)
print("Enlace:", pagina.resolve().as_uri())
if ABRIR_VISOR:
    print("Abriendo en el navegador…" if visor.abrir(pagina) else "No se pudo abrir el navegador: abre el enlace a mano.")

Visor: /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0/output/20261007T095415Z/visor.html
Enlace: file:///Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0/output/20261007T095415Z/visor.html
Abriendo en el navegador…
